# JEPX Scraping
**Scrape historical JEPX data**

## 1. Import relative library

In [1]:
import os
import time
import requests
import polars as pl
from bs4 import BeautifulSoup
from typing import Literal
from pathlib import Path
import numpy as np

## 2. Define constraint variable

In [2]:
STORAGE_PATH = r"/workspace/src/stg/data_lake/eec"

PARENT_URL = "https://public.eex-group.com/ecc/risk-management/reports-files/"

In [3]:
res = requests.get(PARENT_URL)
soup = BeautifulSoup(res.content, "html.parser")
csv_links = soup.find_all("a", href=True)
print(csv_links)

[<a href="..">..</a>, <a href="20251117_MarginBuffer_20251118.csv">20251117_MarginBuffer_20251118.csv</a>, <a href="20251117_intercommodityspreads_20251118.csv">20251117_intercommodityspreads_20251118.csv</a>, <a href="20251117_scanningranges_20251118.csv">20251117_scanningranges_20251118.csv</a>, <a href="20251118_MarginBuffer_20251119.csv">20251118_MarginBuffer_20251119.csv</a>, <a href="20251118_intercommodityspreads_20251119.csv">20251118_intercommodityspreads_20251119.csv</a>, <a href="20251118_scanningranges_20251119.csv">20251118_scanningranges_20251119.csv</a>, <a href="20251119_MarginBuffer_20251120.csv">20251119_MarginBuffer_20251120.csv</a>, <a href="20251119_intercommodityspreads_20251120.csv">20251119_intercommodityspreads_20251120.csv</a>, <a href="20251119_scanningranges_20251120.csv">20251119_scanningranges_20251120.csv</a>, <a href="20251120_MarginBuffer_20251121.csv">20251120_MarginBuffer_20251121.csv</a>, <a href="20251120_intercommodityspreads_20251121.csv">20251120

In [9]:
for link in csv_links:
    file_name = link.text.strip()
    if file_name.endswith('.csv'):
        time.sleep(np.random.uniform(1,5))
        file_url = PARENT_URL  + file_name
        print(f"Downloading {file_name}...")
        df = pl.read_csv(file_url)

        file_path = os.path.join(STORAGE_PATH, file_name)
        print(f"Saving {file_name}...")
        df.write_parquet(file_path.replace('.csv', '.parquet'))
    else:
        continue

Saving 20251117_MarginBuffer_20251118.csv...


KeyboardInterrupt: 

## 3. Define custom functions

In [3]:
def scrape_jepx_data(end_point:str, file_name:str, hist_year:str) -> bytes:
    """_summary_.

    Args:
        end_point (str): _description_
        file_name (str): _description_
        hist_year (str): _description_

    Returns:
        bytes: _description_
    """
    time_stamp = int(time.time() * 1000)

    url = f"https://www.jepx.jp/_download.php?timestamp={time_stamp}"

    headers = {
        "Content-Type" : "application/x-www-form-urlencoded",
        "Referer" : f"{PARENT_URL}/{end_point}/"
    }

    data = {
        "dir": file_name,
        "file": f"{file_name}_{hist_year}.csv"
    }

    response = requests.post(url, headers=headers, data=data)

    return response.content

In [4]:
def save_csv_as_parquet(csv_bytes:bytes, output_path: Path | str, *, compression: Literal["zstd","snappy","gzip","lz4","uncompressed"]="zstd", encoding:str = "cp932") -> Path:
    """_summary_

    Args:
        csv_bytes (bytes): _description_
        output_path (Path | str): _description_
        compression (Literal["zstd","snappy","gzip","lz4","uncompressed"], optional): _description_. Defaults to "zstd".
        encoding (str, optional): _description_. Defaults to "cp932".

    Returns:
    """

    output_path = Path(output_path)

    df = pl.read_csv(csv_bytes, encoding=encoding)

    df.write_parquet(output_path, compression=compression)

    print(f"✓ Saved {len(df):,} rows to {output_path}")
    print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

    return output_path

## 4. Scrape JEPX data

In [5]:
hist_year = 2025

for data_key in DATA_DICT_KEYS:
    end_point = DATA_DICT[data_key]
    print(f"Scraping {data_key} data for year {hist_year}...")
    try:
        time.sleep(np.random.uniform(1,5))
        csv_bytes = scrape_jepx_data(data_key, end_point, str(hist_year))

        output_dir = Path(STORAGE_PATH) / data_key

        output_path = output_dir / f"{end_point}_{hist_year}.parquet"

        save_csv_as_parquet(csv_bytes, output_path)
    except Exception as e:
        print(f"✗ Failed to scrape {data_key} data for year {hist_year}: {e}")
        continue

Scraping spot data for year 2025...
✓ Saved 11,616 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2025.parquet
  File size: 0.48 MB
Scraping intraday data for year 2025...
✓ Saved 11,570 rows to /workspace/src/stg/data_lake/jepx/intraday/intraday_2025.parquet
  File size: 0.18 MB
Scraping forward data for year 2025...
✓ Saved 186 rows to /workspace/src/stg/data_lake/jepx/forward/forward_2025.parquet
  File size: 0.00 MB
Scraping transmission_rights data for year 2025...
✓ Saved 210 rows to /workspace/src/stg/data_lake/jepx/transmission_rights/transmission_rights_2025.parquet
  File size: 0.01 MB
Scraping baseload data for year 2025...
✓ Saved 12 rows to /workspace/src/stg/data_lake/jepx/baseload/baseload_2025.parquet
  File size: 0.00 MB
Scraping fit_fip data for year 2025...
✗ Failed to scrape fit_fip data for year 2025: HTTPSConnectionPool(host='www.jepx.jp', port=443): Max retries exceeded with url: /_download.php?timestamp=1764255438517 (Caused by NameResolutionError("

In [6]:
def scrape_jepx_nf_data(hist_year:str, *, file_name:str="nf_summary") -> bytes:
    """_summary_.

    Args:
        end_point (str): _description_
        file_name (str): _description_
        hist_year (str): _description_

    Returns:
        bytes: _description_
    """
    time_stamp = int(time.time() * 1000)

    NF_PARENT_URL = "https://www.jepx.jp/nonfossil/market-data/"

    url = f"https://www.jepx.jp/_download.php?timestamp={time_stamp}"

    headers = {
        "Content-Type" : "application/x-www-form-urlencoded",
        "Referer" : f"{NF_PARENT_URL}/"
    }

    data = {
        "dir": file_name,
        "file": f"{file_name}_{hist_year}.csv"
    }

    response = requests.post(url, headers=headers, data=data)

    return response.content

In [7]:
print(f"Scraping nonfossil data for year {hist_year}...")
try:
    time.sleep(np.random.uniform(1,5))

    csv_bytes = scrape_jepx_nf_data(str(hist_year))

    output_dir = Path(STORAGE_PATH) / "nonfossil"

    output_path = output_dir / f"nf_summary_{hist_year}.parquet"

    save_csv_as_parquet(csv_bytes, output_path)
except Exception as e:
    print(f"✗ Failed to scrape {data_key} data for year {hist_year}: {e}")

Scraping nonfossil data for year 2025...
✓ Saved 5 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_summary_2025.parquet
  File size: 0.00 MB
